In [1]:
import os

from langchain_experimental.utilities import PythonREPL
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import Tool

from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
## Start by creating an instance of the AzureChatOpenAI class.
chat = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    temperature=0,
)

In [3]:
# Tool: Python REPL for heavy math
python_tool = Tool(
    name="Python Calculator",
    func=PythonREPL().run,
    description="Useful for large computations (e.g., powers beyond 20). Input must be valid Python."
)

In [4]:
# Example query
query = "Calculate 10000 * 12**2 - 12 + log(10)"

tools = [python_tool]  # you can add wikipedia tool too

In [ ]:
# Other tools (optional, like Wikipedia)
#!pip install wikipedia
#import wikipedia
#from langchain_community.agent_toolkits.load_tools import load_tools
#other_tools = load_tools(["wikipedia"], llm=chat)
# Combine tools
#tools = [python_tool] + other_tools

In [5]:
from langsmith import Client
prompt = Client().pull_prompt("hwchase17/react")


In [6]:
import langchain
import langchain_core
import langchain_community
import pydantic

print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print("langchain_community:", langchain_community.__version__)
print("pydantic:", pydantic.__version__)

langchain: 1.2.18
langchain_core: 1.3.3
langchain_community: 0.4.1
pydantic: 2.13.2


In [8]:
from langchain_classic.agents import create_react_agent, AgentExecutor

agent = create_react_agent(
    llm=chat,
    tools=tools,
    prompt=prompt
)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)


In [9]:
# Query example
result = executor.invoke({"input": "Calculate 10000 * 12**2 - 12 + log(10)"})
print(result["output"])



> Entering new AgentExecutor chain...


Python REPL can execute arbitrary code. Use with caution.


Question: Calculate 10000 * 12**2 - 12 + log(10)
Thought: I need to calculate the expression step-by-step. The expression is 10000 * 12^2 - 12 + log(10). The log here is likely the natural logarithm (ln). I will calculate 12 squared, multiply by 10000, subtract 12, and then add the natural logarithm of 10.
Action: Python Calculator
Action Input: import math; 10000 * 12**2 - 12 + math.log(10)Thought: I have calculated the expression 10000 * 12**2 - 12 + log(10) using the natural logarithm. The result is ready.
Final Answer: 143988.02585092985

> Finished chain.
143988.02585092985


In [10]:
# Query example
result = executor.invoke({"input": query})
print(result["output"])



> Entering new AgentExecutor chain...
Question: Calculate 10000 * 12**2 - 12 + log(10)
Thought: I need to calculate the expression step-by-step. The expression is 10000 * 12^2 - 12 + log(10). The log here is likely the natural logarithm (ln). I will calculate 12 squared, multiply by 10000, subtract 12, and then add the natural logarithm of 10.
Action: Python Calculator
Action Input: import math; 10000 * 12**2 - 12 + math.log(10)Thought: I have calculated the expression 10000 * 12**2 - 12 + log(10) using the natural logarithm. The result is ready.
Final Answer: 143988.02585092985

> Finished chain.
143988.02585092985


In [ ]:
#Using newer approach
from langchain_openai import AzureChatOpenAI
from langchain_core.runnables import RunnableSequence

In [11]:
chat = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT"),
    temperature=0,
)

# Example tools as Python functions
def echo_tool(input_text: str) -> str:
    return f"Echo: {input_text}"

def reverse_tool(input_text: str) -> str:
    return input_text[::-1]

# Agent sequence
def agent_step(user_input):
    # Decide which tool to use
    if "reverse" in user_input.lower():
        return reverse_tool(user_input)
    else:
        return echo_tool(user_input)

# Memory
chat_history = []

def add_memory(role, msg):
    chat_history.append({"role": role, "content": msg})

# Multi-step runnable sequence
def remember_step(user_input):
    add_memory("user", user_input)
    return user_input

def llm_step(user_input):
    # Minimal safe prompt
    prompt = f"You are a helpful assistant. User says: {user_input}. Respond helpfully."
    response = chat.invoke(prompt)
    add_memory("assistant", response.content)
    tool_result = agent_step(user_input)
    return f"{response.content}\nTool result: {tool_result}"

from langchain_core.runnables import RunnableSequence
sequence = RunnableSequence(remember_step, llm_step)

output = sequence.invoke("Hello agent, reverse this text!")
print(output)

Hello! Sure, here is your reversed text: "!txet siht esrever ,tnega olleH"  
If you need help with anything else, just let me know!
Tool result: !txet siht esrever ,tnega olleH
